# Behavision inactivity pipeline on Google Colab

Runs every video in a Google Drive folder and writes results back to Drive.

1. **Runtime > Change runtime type > GPU**, then run the cells top to bottom.
2. Upload the videos to `MyDrive/behavision/videos_ITI/` first (or change `VIDEO_DIR` below).
3. If Colab disconnects, reconnect and run all cells again: finished videos are
   skipped and an unfinished one continues from its last saved chunk.
   Keep this browser tab open while it runs.

In [ ]:
# 1. GPU check: a CPU-only runtime would take days.
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# 2. Google Drive (holds the videos and all results)
from google.colab import drive
drive.mount('/content/drive')

VIDEO_DIR = '/content/drive/MyDrive/behavision/videos_ITI'
RESULTS_DIR = '/content/drive/MyDrive/behavision/results'
import glob, os
os.makedirs(RESULTS_DIR, exist_ok=True)
videos = sorted(glob.glob(f'{VIDEO_DIR}/*.avi') + glob.glob(f'{VIDEO_DIR}/*.mp4'))
print(len(videos), 'videos'); print(*videos, sep='\n')

In [ ]:
# 3. Code and dependencies (about 2 minutes; uses Colab's own CUDA PyTorch)
%cd /content
!test -d behavision || git clone -q https://github.com/djism/behavision.git
%cd /content/behavision
!git pull -q
!SAM2_BUILD_CUDA=0 pip install -q -e ".[tracking]"

In [ ]:
# 4. Run every video. Results: RESULTS_DIR/<video name>/ (bouts, summary,
#    overview chart, review page, annotated.mp4). Roughly hours per video.
import subprocess, os, collections
env = {**os.environ, 'TQDM_DISABLE': '1', 'TOKENIZERS_PARALLELISM': 'false'}
for video in videos:
    name = os.path.splitext(os.path.basename(video))[0]
    out = f'{RESULTS_DIR}/{name}'
    if os.path.exists(f'{out}/annotated.mp4'):
        print('Already done:', name); continue
    print('=== ', name, flush=True)
    cmd = ['behavision-run', video, '--config', 'configs/iti_homecage.yaml', '--out', out,
           '--work-dir', f'/content/work/{name}', '--annotated-video']
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    tail = collections.deque(maxlen=40)
    for line in proc.stdout:
        tail.append(line)
        if line.startswith('['):  # timestamped progress lines only
            print(line, end='', flush=True)
    if proc.wait() != 0:
        print(''.join(tail))
        raise RuntimeError(f'{name} failed; see the output above')

In [ ]:
# 5. One table for all videos: all_videos_summary.csv and all_bouts.csv in RESULTS_DIR
!behavision-combine "$RESULTS_DIR"
import pandas as pd
pd.read_csv(f'{RESULTS_DIR}/all_videos_summary.csv')[['video_id', 'inactive_s', 'bouts', 'uncertain_s', 'tracked_fraction', 'qc_warnings']]